# Iteration 03 — Filtering & DEITA-style Selection on Bangla-Instruct

This notebook replaces Stage 2–4 of `iteratinon02.ipynb` and implements Tasks 2–4 of the plan:
a hygiene pre-filter applied to the whole pool, then model-based scores, then the subsets for
experimental arms A–E.

| Stage | What it does | Output |
|---|---|---|
| 1 | Load **all 342,391** Bangla-Instruct rows, tag domain, sample the pool + a held-out set | `pool/pool_raw.parquet` |
| 2 | Bengali-aware normalisation (keeps ZWJ/ZWNJ, code indentation, LaTeX) | — |
| 3 | Heuristic filters with a **reason per dropped row** + a per-domain bias check | `reports/filter_report.csv` |
| 4 | Exact dedup + MinHash-LSH near-dedup | — |
| 5 | Benchmark decontamination (13-gram) against the eval sets and the held-out prompts | `pool/pool_clean.parquet` |
| 6 | *(GPU)* Perplexity + IFD with the base model | `scores/ppl_ifd.parquet` |
| 7 | *(GPU)* LLM-judge complexity **c** and quality **q** (expected score from digit probabilities) | `scores/judge.parquet` |
| 8 | *(GPU)* Embeddings + sanity check | `scores/embeddings.npy` |
| 9 | Build arms A–E (+ optional F) and per-arm diagnostics | `arms/*.jsonl`, `reports/arm_diagnostics.csv` |
| 10 | Blind spot-check sheet, timings, cost extrapolation, manifest | `reports/` |

**Run `RUN_MODE="pilot"` first** (1,000 rows — this *is* the Task 4 feasibility pilot), read the
pilot report in Stage 7/8, then switch to `"full"`.

Why the old pipeline is not reused as-is (findings from profiling the real file):
* the HF file is several sub-corpora concatenated (Alpaca-style CoT → FLAN-style translated tasks →
  native reading-comprehension → translated competition math in LaTeX). The old language check
  counted LaTeX as English and silently removed ~30% of the math rows;
* `ZERO_WIDTH` stripped ZWJ/ZWNJ (meaningful in Bengali, e.g. র‍্য), `MULTISPACE` flattened Python
  indentation, `HTML_TAG` deleted `<int>` / `a < b > c`;
* the 0–100 quality score saturated (median 100) so it could not rank anything;
* keyword task-typing put 74% of rows in "question_answering" because `"কী"` matches almost everything.

In [ ]:
%pip install -q datasets huggingface_hub ftfy rapidfuzz datasketch sentence-transformers pyarrow matplotlib scipy
# Stage 7 with JUDGE_BACKEND="vllm" also needs:  %pip install -q vllm   (restart the runtime afterwards)

In [ ]:
import os, re, io, json, math, time, random, hashlib, unicodedata, html
import importlib.util
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # before torch touches CUDA

# --- platform: Kaggle / Colab / local ----------------------------------------------------------------
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = os.path.exists("/content") and not ON_KAGGLE

def hf_login():
    """Read HF_TOKEN from Kaggle Secrets / Colab secrets (needed for the gated meta-llama weights)."""
    token = os.environ.get("HF_TOKEN")
    try:
        if not token and ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        elif not token and ON_COLAB:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
    except Exception as e:
        print("no HF_TOKEN secret found:", type(e).__name__)
    if token:
        os.environ["HF_TOKEN"] = token
        from huggingface_hub import login
        login(token=token)

hf_login()

CONFIG = {
    "RUN_MODE":        "pilot",      # "pilot" (Task 4, ~1K rows) | "full"
    "PILOT_SIZE":      1_000,
    "POOL_SIZE":       100_000,      # the paper's size; the HF file actually holds 342,391 rows
    "POOL_SAMPLING":   "stratified", # stratified by domain (keeps the file's mix exactly) | "uniform"
    "HELDOUT_SIZE":    500,          # removed BEFORE any filtering/selection; never trained on
    "SUBSET_SIZE":     40_000,       # arms B–E
    "SEED":            42,

    "INSTRUCT_REPO":   "md-nishat-008/Bangla-Instruct",   # licence on the Hub: MIT (not cc-by-4.0)
    "INCLUDE_SECONDARY": False,      # iteration02 open-QnA/Wikipedia rows. Keep False for the A–E study:
    "SECONDARY_PATH":  "/content/bangla_instruct_30k/raw/secondary_raw.jsonl",  # mixing sources adds a 2nd variable

    # --- Stage 3 heuristics ---
    "MIN_INSTR_CHARS": 8,
    "MIN_RESP_CHARS":  20,           # relaxed to 2 for domains where short answers are correct
    "MAX_RESP_CHARS":  12_000,
    "MIN_BN_PROSE_RATIO": 0.60,      # Bengali share of letters AFTER removing code / LaTeX / URLs
    "MAX_DUP_LINE_FRAC":  0.50,      # Gopher-style repetition limits, loosened: at Gopher's 0.15–0.30 step-by-step
    "MAX_TOP2GRAM_FRAC":  0.20,      # answers that quote the question get flagged. Not applied to math, whose
    "MAX_DUP5GRAM_FRAC":  0.50,      # LaTeX-stripped skeleton ("সমাধান করি: …") is repetitive by design.
    "MAX_LATIN_RUN":      8,         # ≥8 consecutive English words in Bengali prose = untranslated text
    "DROP_TRANSLATIONESE": True,

    # --- Stage 4 dedup ---
    "MINHASH_PERM": 128, "MINHASH_THRESHOLD": 0.80, "SHINGLE": 5,

    # --- Stage 5 decontamination ---
    "DECON_N": 13, "DECON_SHORT_N": 8,
    "DECON_TEMPLATE_DF": 10,         # an n-gram shared by more pool rows than this is a prompt template, not a leak
    "MHUMANEVAL_BN_URL": "https://raw.githubusercontent.com/mraihan-gmu/mHumanEval-Benchmark/main/mHumanEval-max/mHumanEval-ben_Beng-python.csv",
    # BEnQA, BanglaQuaD and PangBench-bn are not on the HF Hub under those names. When you obtain
    # them, add (name, path_or_url, [text columns]) here so they are decontaminated too.
    "EXTRA_BENCH_FILES": [],

    # --- Stages 6–8 (GPU) ---
    "BASE_MODEL":          "meta-llama/Llama-3.2-1B",   # gated: accept the licence + set HF_TOKEN
    "BASE_MODEL_FALLBACK": "unsloth/Llama-3.2-1B",      # ungated mirror of the same weights
    "RUN_PPL_IFD":   True,
    # "vllm" (fast) when installed, else "hf". Not vLLM on Kaggle: installing it needs a runtime restart,
    # which a pushed (batch) kernel cannot do.
    "JUDGE_BACKEND": "vllm" if (importlib.util.find_spec("vllm") and not ON_KAGGLE) else "hf",
    "JUDGE_MODEL":   "Qwen/Qwen2.5-7B-Instruct",  # open-weight on purpose: Bangla-Instruct was generated by
                                                  # GPT-4o + Claude-3.5, a same-family judge would be circular.
                                                  # Compare e.g. google/gemma-2-9b-it in the pilot.
    "JUDGE_MAX_INSTR_CHARS": 1_200,
    "JUDGE_MAX_RESP_CHARS":  2_000,
    "JUDGE_MIN_MASS":        0.5,    # probability on digits 1–6 below this = judge ignored the format
    "EMBED_MODEL":   "BAAI/bge-m3",
    # DEITA: add a pair only if its cos-sim to the nearest already-selected pair is < τ. DEITA's 0.9 was for a
    # different embedding model; with bge-m3 the 1K pilot had no pair above 0.89, so τ=0.9 never fired (C == E).
    # "auto" = this quantile of each row's nearest-neighbour similarity in the actual pool.
    "TAU_SIM":       "auto",
    "TAU_NN_QUANTILE": 0.90,

    "USE_DRIVE": False,              # True = write outputs to Google Drive (survives Colab disconnects)
    "OUT_DIR":   "/kaggle/working/bangla_filtering" if ON_KAGGLE else "/content/bangla_filtering",
}

FULL_POOL_SIZE = CONFIG["POOL_SIZE"]          # used to extrapolate pilot cost
if CONFIG["RUN_MODE"] == "pilot":
    CONFIG.update(POOL_SIZE=CONFIG["PILOT_SIZE"], HELDOUT_SIZE=50,
                  SUBSET_SIZE=int(CONFIG["PILOT_SIZE"] * CONFIG["SUBSET_SIZE"] / FULL_POOL_SIZE))

if CONFIG["USE_DRIVE"] and ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CONFIG["OUT_DIR"] = "/content/drive/MyDrive/bangla_filtering"

random.seed(CONFIG["SEED"]); np.random.seed(CONFIG["SEED"])
OUT = Path(CONFIG["OUT_DIR"]) / CONFIG["RUN_MODE"]
for sub in ["pool", "scores", "arms", "reports"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

TIMINGS = {}
@contextmanager
def stage(name, n_rows=None):
    t0 = time.time()
    yield
    dt = time.time() - t0
    TIMINGS[name] = {"seconds": round(dt, 1), "rows": n_rows}
    print(f"[{name}] {dt:,.1f}s" + (f" for {n_rows:,} rows" if n_rows else ""))

print(json.dumps({k: CONFIG[k] for k in ["RUN_MODE", "POOL_SIZE", "HELDOUT_SIZE", "SUBSET_SIZE"]}), "->", OUT)

---
## Stage 1 — Load, tag domain, sample pool + held-out
The domain tag is only used for **stratified sampling and bias reporting**, never to drop rows.
Rules are ordered specific → generic (the old notebook checked `"কী"` before code/math).

In [ ]:
from datasets import load_dataset

with stage("load"):
    raw = load_dataset(CONFIG["INSTRUCT_REPO"], split="train").to_pandas()
raw = raw.rename(columns={"response": "output"})
raw["orig_idx"] = np.arange(len(raw))          # file order == sub-corpus order
raw = raw.dropna(subset=["instruction", "output"])
raw["instruction"] = raw["instruction"].astype(str)
raw["output"] = raw["output"].astype(str)
raw["source_type"] = "bangla_instruct"
raw["id"] = ["bi" + str(i).zfill(6) for i in raw["orig_idx"]]

if CONFIG["INCLUDE_SECONDARY"] and Path(CONFIG["SECONDARY_PATH"]).exists():
    sec = pd.read_json(CONFIG["SECONDARY_PATH"], lines=True)
    sec["instruction"] = sec["instruction"] + np.where(sec["input"].str.len() > 0, "\n\n" + sec["input"], "")
    sec["orig_idx"] = -1
    sec["id"] = ["sec" + str(i).zfill(6) for i in range(len(sec))]
    raw = pd.concat([raw, sec[["id", "instruction", "output", "source_type", "orig_idx"]]], ignore_index=True)

print(f"{len(raw):,} rows")

In [ ]:
DOMAIN_RULES = [   # (name, pattern, where) — first match wins
    ("math",        r"\\boxed|\\frac|\\sqrt|\\\(|\\\[|\$[^$\n]{1,80}\$|সমীকরণ|গণনা কর|সমাধান কর|মান নির্ণয়", "both"),
    ("code",        r"```|\bdef |\bimport |#include|console\.log|\bSELECT\b|public static|কোড|প্রোগ্রাম|ফাংশন|স্ক্রিপ্ট|রেজেক্স|\b(?:python|Python|Java|SQL|HTML|CSS|JavaScript)\b", "both"),
    ("translation", r"অনুবাদ|\b[Tt]ranslate\b|ইংরেজিতে|বাংলায় রূপান্তর", "instr"),
    ("mcq_classification", r"Choices:|Options:|OPTIONS|বিকল্প(?:গুলো|সমূহ)?\s*:|\n\s*[কখগঘ][\)\.]|\n\s*[A-D][\)\.]\s|হ্যাঁ বা না|সত্য না মিথ্যা|শ্রেণ[ীি]বদ্ধ|শনাক্ত কর|চিহ্নিত কর", "instr"),
    ("summarization", r"সারসংক্ষেপ|সারাংশ|সংক্ষেপে|সংক্ষিপ্তসার", "instr"),
    ("creative",    r"গল্প|কবিতা|রচনা|চিঠি|ছড়া|শিরোনাম|স্লোগান|সংলাপ", "instr"),
    ("reading_comp", None, "instr"),
    ("list_brainstorm", r"তালিকা|উপায়|টিপস|পরামর্শ|উদাহরণ দা|উদাহরণ দি", "instr"),
    ("open_qa",     r"\?|কী|কি |কেন|কীভাবে|কিভাবে|কোথায়|কখন|কত|কোন|কে ", "instr"),
]
_DOMAIN_RX = [(n, re.compile(p) if p else None, w) for n, p, w in DOMAIN_RULES]
SHORT_ANSWER_DOMAINS = {"mcq_classification", "reading_comp", "open_qa", "translation"}

def tag_domain(instr, resp):
    both = instr + "\n" + resp[:2000]
    for name, rx, where in _DOMAIN_RX:
        if rx is None:                                   # reading comprehension: long passage + question
            if len(instr) > 500 and ("?" in instr[-300:] or "প্রশ্ন" in instr):
                return name
            continue
        if rx.search(both if where == "both" else instr):
            return name
    return "other"

with stage("domain_tag", len(raw)):
    raw["domain"] = [tag_domain(i, o) for i, o in zip(raw["instruction"], raw["output"])]
raw["segment"] = pd.cut(raw["orig_idx"], 10, labels=[f"s{i}" for i in range(10)])  # position in file

print(raw["domain"].value_counts().to_string())
display(pd.crosstab(raw["segment"], raw["domain"], normalize="index").round(2))

In [ ]:
n_take = min(CONFIG["POOL_SIZE"] + CONFIG["HELDOUT_SIZE"], len(raw))
if CONFIG["POOL_SAMPLING"] == "stratified":
    sampled = raw.groupby("domain").sample(frac=n_take / len(raw), random_state=CONFIG["SEED"])
else:
    sampled = raw.sample(n=n_take, random_state=CONFIG["SEED"])
sampled = sampled.sample(frac=1, random_state=CONFIG["SEED"]).reset_index(drop=True)

heldout = sampled.iloc[:CONFIG["HELDOUT_SIZE"]].copy().reset_index(drop=True)
pool = sampled.iloc[CONFIG["HELDOUT_SIZE"]:].copy().reset_index(drop=True)
pool.to_parquet(OUT / "pool" / "pool_raw.parquet", index=False)
print(f"pool {len(pool):,} | held-out {len(heldout):,}")

---
## Stage 2 — Bengali-aware normalisation
* `ftfy` mojibake repair + **NFC**: ~41% of responses mix precomposed য় (U+09DF) with the decomposed য + ়
  form; NFC makes them one codepoint sequence (same for ড়/ঢ়). Curly quotes are left alone.
* keep **ZWJ/ZWNJ** when they sit inside Bengali text (র‍্যাব, conjunct control); drop other zero-widths
* legacy khanda-ta `ত্‍` → `ৎ`
* collapse runs of spaces **between** words only — leading indentation (code, nested lists) survives
* strip only real HTML tags (`<br>`, `<p>` …) and never inside fenced code; URLs are kept, not masked

In [ ]:
import ftfy

CODE_FENCE = re.compile(r"```.*?(?:```|\Z)", re.S)
ZW_DROP    = re.compile(r"[​⁠﻿­]")
ZWJ_STRAY  = re.compile(r"(?<![ঀ-৿])[‌‍]|[‌‍](?![ঀ-৿])")
KHANDA_TA  = re.compile("ত্‍")
HTML_KNOWN = re.compile(r"</?(?:p|br|div|span|b|i|u|a|ul|ol|li|strong|em|font|table|tr|td|th|h[1-6])(?:\s[^<>]*)?/?>", re.I)
INNER_WS   = re.compile(r"(?<=\S)[ \t 　]{2,}")
TRAIL_WS   = re.compile(r"[ \t]+\n")

def _clean_prose(t):
    t = html.unescape(t)
    t = HTML_KNOWN.sub(" ", t)
    t = INNER_WS.sub(" ", t)
    return TRAIL_WS.sub("\n", t)

def normalize_bn(s):
    if not isinstance(s, str):
        return ""
    s = ftfy.fix_text(s, normalization="NFC", uncurl_quotes=False)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = KHANDA_TA.sub("ৎ", s)
    s = ZW_DROP.sub("", s)
    s = ZWJ_STRAY.sub("", s)
    parts, last = [], 0
    for m in CODE_FENCE.finditer(s):                 # leave fenced code byte-identical
        parts.append(_clean_prose(s[last:m.start()]))
        parts.append(m.group(0))
        last = m.end()
    parts.append(_clean_prose(s[last:]))
    s = "".join(parts)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def normalize_frame(frame):
    frame = frame.copy()
    for col in ["instruction", "output"]:
        frame[col] = [normalize_bn(x) for x in frame[col]]
    return frame

with stage("normalize", len(pool)):
    changed = pool[["instruction", "output"]].copy()
    pool = normalize_frame(pool)
    heldout = normalize_frame(heldout)
print("rows changed by normalisation:",
      f"instruction {(changed['instruction'] != pool['instruction']).mean():.2%},",
      f"output {(changed['output'] != pool['output']).mean():.2%}")
del changed

---
## Stage 3 — Heuristic quality filters (every drop gets a reason)

| Reason | Rule |
|---|---|
| `length` | instruction < 8 chars; response < 20 chars (2 for QA/MCQ/translation) or > 12K |
| `mojibake` | U+FFFD or `à¦…`-style residue left after ftfy |
| `template_artifact` | `নোইনপুট` / `<noinput>` / `### Instruction:` / chat-special-token leaks |
| `identity_refusal` | "একটি এআই ভাষা মডেল হিসেবে…", "As an AI", "I am ChatGPT/Claude", Bengali refusals |
| `language` | Bengali share of letters in **prose** (code, LaTeX, URLs removed) < 0.60; for translation tasks one side may be English |
| `translationese` | ≥ 8 consecutive English words in Bengali prose, or English template words (`Choices:`, `Premise:`) |
| `echo` | response ≈ instruction (fuzz ratio > 90) |
| `truncated` | unclosed ``` fence, or a long response ending mid-word (math `\boxed{}` / `$` endings are fine). Rows are first **repaired** by cutting back to the last full sentence when that keeps ≥ 60% of the text; only the rest are dropped |
| `char_spam` | one character repeated 10+ times (markdown rules / digits excluded) |
| `no_space_wall` | ≥ 50 Bengali code points with no space — broken segmentation |
| `repetition` | Gopher-style: duplicate-line share, top-bigram share, duplicated-5-gram share (prose only, ≥ 50 words, not math) |

In [ ]:
from rapidfuzz import fuzz

BN_CH     = re.compile(r"[ঀ-৿]")
LATIN_CH  = re.compile(r"[A-Za-z]")
WORD      = re.compile(r"[ঀ-৿‌‍A-Za-z0-9]+")
MATH_SPAN = re.compile(r"\$\$.*?\$\$|\$[^$\n]+\$|\\\(.*?\\\)|\\\[.*?\\\]|\\[A-Za-z]+(?:\{[^{}]*\})*", re.S)
INLINE_CODE = re.compile(r"`[^`\n]+`")
URL_RE    = re.compile(r"https?://\S+|www\.\S+")
LATIN_WORD = re.compile(r"[A-Za-z][A-Za-z'’\-]*[.,;:!?]?")

MOJIBAKE  = re.compile(r"�|à¦|à§|Ã[\x80-\xbf]|â€")
ARTIFACT  = re.compile(r"নো\s?ইনপুট|<\s*no\s*_?input\s*>|\bno\s?input\b|(?:কোন|কোনো)\s+ইনপুট\s+নেই|ইনপুট\s*:\s*(?:নেই|N/?A)?\s*\Z"
                       r"|###\s*(?:Instruction|Response|Input)\b|<\|(?:im_start|im_end|endoftext|eot_id|start_header_id)\|>", re.I | re.M)
IDENTITY  = re.compile(r"(?:একটি|একজন)\s+(?:এআই|AI|কৃত্রিম\s+বুদ্ধিমত্তা(?:র)?)\s*(?:ভাষা\s*)?(?:মডেল|সহকারী|চ্যাটবট)\s*(?:হিসেবে|হিসাবে)"
                       r"|\bas an ai\b|\bi(?:'m| am) (?:sorry|unable|not able)\b|(?:আমি|আমাকে)\s+(?:ChatGPT|GPT-?4o?|Claude)\b"
                       r"|(?:OpenAI|Anthropic)\s+(?:আমাকে|আমায়)"
                       r"|আমি\s+দুঃখিত[,]?\s*(?:কিন্তু|তবে)?.{0,40}?(?:পারব\s+না|পারি\s+না|সম্ভব\s+নয়|সক্ষম\s+নই)", re.I)
EN_TEMPLATE = re.compile(r"\b(?:Choices|Options|OPTIONS|Answer|Question|Premise|Hypothesis|Passage|Context)\s*:")
CHAR_SPAM = re.compile(r"([^\s\-=_*#.~০-৯0-9|])\1{9,}")
NO_SPACE_WALL = re.compile(r"[ঀ-৿‌‍]{50,}")
END_OK    = tuple("।॥?!.…\"'”’)]}>*`|:;$%০১২৩৪৫৬৭৮৯0123456789")

def prose_only(s):
    for rx in (CODE_FENCE, MATH_SPAN, INLINE_CODE, URL_RE):
        s = rx.sub(" ", s)
    return s

def bn_prose_ratio(s):
    p = prose_only(s)
    b, l = len(BN_CH.findall(p)), len(LATIN_CH.findall(p))
    return b / (b + l) if (b + l) else 1.0          # pure math / code: nothing to judge

def max_latin_run(s):
    best = run = 0
    for tok in prose_only(s).split():
        run = run + 1 if LATIN_WORD.fullmatch(tok) else 0
        best = max(best, run)
    return best

def repetition_stats(text):
    p = prose_only(text)
    lines = [l.strip() for l in p.split("\n") if l.strip()]
    dup_line = 1 - len(set(lines)) / len(lines) if len(lines) >= 4 else 0.0
    w = WORD.findall(p)
    if len(w) < 50:
        return dup_line, 0.0, 0.0
    total = sum(map(len, w))
    (b1, b2), cnt = Counter(zip(w, w[1:])).most_common(1)[0]
    top2 = cnt * (len(b1) + len(b2)) / total
    grams = [tuple(w[i:i + 5]) for i in range(len(w) - 4)]
    c5 = Counter(grams)
    dup = np.zeros(len(w), bool)
    for i, g in enumerate(grams):
        if c5[g] > 1:
            dup[i:i + 5] = True
    dup5 = sum(len(x) for x, d in zip(w, dup) if d) / total
    return dup_line, top2, dup5

def looks_truncated(resp):
    t = resp.rstrip()
    if t.count("```") % 2 == 1:
        return True
    return len(t) >= 60 and not t.endswith(END_OK)

SENT_END = re.compile(r"[।॥?!.](?=\s|$)")

def repair_truncated(resp, keep_frac=0.6):
    """Cut a response that stops mid-sentence back to its last complete sentence.
    Returns the repaired text, or None when that would lose too much (or a code fence is open)."""
    t = resp.rstrip()
    if t.count("```") % 2 == 1:
        return None
    ends = [m.end() for m in SENT_END.finditer(t)]
    if not ends or ends[-1] < keep_frac * len(t):
        return None
    return t[:ends[-1]]

FILTER_ORDER = ["length", "mojibake", "template_artifact", "identity_refusal", "language",
                "translationese", "echo", "truncated", "char_spam", "no_space_wall", "repetition"]

def heuristic_flags(instr, resp, domain):
    C = CONFIG
    min_resp = 2 if domain in SHORT_ANSWER_DOMAINS else C["MIN_RESP_CHARS"]
    ri, ro = bn_prose_ratio(instr), bn_prose_ratio(resp)
    if domain == "translation":      # one side is legitimately English
        lang_ok = max(ri, ro) >= C["MIN_BN_PROSE_RATIO"]
    elif domain == "code":           # responses are often unfenced code (pilot: 19% of code rows wrongly dropped)
        lang_ok = ri >= C["MIN_BN_PROSE_RATIO"]
    else:
        lang_ok = min(ri, ro) >= C["MIN_BN_PROSE_RATIO"]
    latin_run = 0 if domain in ("translation", "code") else max(max_latin_run(instr), max_latin_run(resp))
    dup_line, top2, dup5 = repetition_stats(resp) if domain != "math" else (0.0, 0.0, 0.0)
    f = {
        "length":            len(instr) < C["MIN_INSTR_CHARS"] or not (min_resp <= len(resp) <= C["MAX_RESP_CHARS"]),
        "mojibake":          bool(MOJIBAKE.search(instr) or MOJIBAKE.search(resp)),
        "template_artifact": bool(ARTIFACT.search(instr) or ARTIFACT.search(resp)),
        "identity_refusal":  bool(IDENTITY.search(resp)),
        "language":          not lang_ok,
        "translationese":    C["DROP_TRANSLATIONESE"] and (latin_run >= C["MAX_LATIN_RUN"]
                                                           or (domain != "translation" and bool(EN_TEMPLATE.search(instr)))),
        "echo":              len(resp) > 30 and fuzz.ratio(instr, resp) > 90,
        "truncated":         looks_truncated(resp),
        "char_spam":         bool(CHAR_SPAM.search(resp)),
        "no_space_wall":     bool(NO_SPACE_WALL.search(prose_only(resp))),
        "repetition":        dup_line > C["MAX_DUP_LINE_FRAC"] or top2 > C["MAX_TOP2GRAM_FRAC"] or dup5 > C["MAX_DUP5GRAM_FRAC"],
    }
    f["drop_reason"] = next((k for k in FILTER_ORDER if f[k]), "")
    f["bn_ratio_instr"], f["bn_ratio_resp"] = round(ri, 4), round(ro, 4)
    return f

def apply_heuristics(frame):
    # repair first: truncation sits mostly in long-form answers (code, lists), so dropping
    # every cut-off row would bias the pool towards short answers
    frame = frame.copy()
    frame["repaired"] = False
    for ix in frame.index[[looks_truncated(o) for o in frame["output"]]]:
        fixed = repair_truncated(frame.at[ix, "output"])
        if fixed is not None:
            frame.at[ix, "output"], frame.at[ix, "repaired"] = fixed, True
    flags = pd.DataFrame([heuristic_flags(i, o, d) for i, o, d in
                          tqdm(zip(frame["instruction"], frame["output"], frame["domain"]), total=len(frame), desc="heuristics")],
                         index=frame.index)
    return pd.concat([frame, flags], axis=1)

with stage("heuristics", len(pool)):
    pool = apply_heuristics(pool)
    heldout = apply_heuristics(heldout)
heldout = heldout[heldout["drop_reason"] == ""].reset_index(drop=True)

In [ ]:
# Reason counts + the bias check: does any filter hit one domain much harder than the rest?
dropped = pool[pool["drop_reason"] != ""]
reason_by_domain = pd.crosstab(pool["domain"], pool["drop_reason"].replace("", "kept"))
drop_rate = (pool["drop_reason"] != "").groupby(pool["domain"]).mean().rename("drop_rate")
report = reason_by_domain.join(drop_rate).sort_values("drop_rate", ascending=False)
report.to_csv(OUT / "reports" / "filter_report.csv")

overall = (pool["drop_reason"] != "").mean()
print(f"heuristics drop {len(dropped):,} / {len(pool):,} ({overall:.2%}) | truncated responses repaired: {int(pool.repaired.sum()):,}")
print(dropped["drop_reason"].value_counts().to_string())
display(report)
biased = drop_rate[drop_rate > 2 * overall]
if len(biased):
    print("⚠ filters hit these domains > 2x the average rate — inspect before trusting the filter:", biased.round(3).to_dict())

# one example per reason, for a quick manual look
for r in FILTER_ORDER:
    ex = dropped[dropped["drop_reason"] == r].head(1)
    if len(ex):
        print(f"\n--- {r} ---\nI: {ex.iloc[0]['instruction'][:200]}\nR: {ex.iloc[0]['output'][-200:]}")

In [ ]:
# PII is masked, not dropped (rare in synthetic data, but cheap to guarantee)
PII = [(re.compile(r"(?<!\d)(?:\+?88)?01[3-9]\d{8}(?!\d)"), "<PHONE>"),
       (re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"), "<EMAIL>")]
pii_hits = 0
for col in ["instruction", "output"]:
    for rx, tag in PII:
        hit = pool[col].str.contains(rx)
        pii_hits += int(hit.sum())
        pool.loc[hit, col] = pool.loc[hit, col].str.replace(rx, tag, regex=True)
print("PII masks applied:", pii_hits)

pool = pool[pool["drop_reason"] == ""].reset_index(drop=True)

---
## Stage 4 — Deduplication
1. **Exact** — hash of the punctuation/space-stripped text (Bengali vowel signs are *kept*; Python's `\W` would delete them).
2. **Near-dup** — MinHash LSH over character 5-gram shingles of instruction + first 1,000 chars of response, Jaccard ≥ 0.8.

Semantic (embedding) dedup is deliberately **not** applied to the whole pool: it would overlap with arm C's
diversity step and blur the C-vs-B comparison. It is reported as a diagnostic in Stage 8.

In [ ]:
from datasketch import MinHash, MinHashLSH

PUNCT_WS = re.compile(r"[\s।॥!-/:-@\[-`{-~“”‘’…–—]+")

def dedup_key(*texts):
    s = unicodedata.normalize("NFC", " ".join(texts)).lower()
    return hashlib.md5(PUNCT_WS.sub("", s).encode("utf-8")).hexdigest()

def shingles(text, k):
    t = re.sub(r"\s+", " ", text)
    return {t[i:i + k] for i in range(max(1, len(t) - k + 1))}

with stage("dedup", len(pool)):
    n0 = len(pool)
    pool["_k_pair"] = [dedup_key(i, o) for i, o in zip(pool["instruction"], pool["output"])]
    pool["_k_instr"] = [dedup_key(i) for i in pool["instruction"]]
    pool = pool.drop_duplicates("_k_pair").drop_duplicates("_k_instr")
    n_exact = n0 - len(pool)

    lsh = MinHashLSH(threshold=CONFIG["MINHASH_THRESHOLD"], num_perm=CONFIG["MINHASH_PERM"])
    keep, dup_pairs = [], []
    texts = dict(zip(pool["id"], pool["instruction"] + " || " + pool["output"].str[:1000]))
    for rid, text in tqdm(texts.items(), desc="minhash"):
        m = MinHash(num_perm=CONFIG["MINHASH_PERM"], seed=CONFIG["SEED"])
        m.update_batch([s.encode("utf-8") for s in shingles(text, CONFIG["SHINGLE"])])
        hits = lsh.query(m)
        if hits:
            dup_pairs.append((hits[0], rid))
            continue
        lsh.insert(rid, m)
        keep.append(rid)
    near_dup_texts = {rid: texts[rid] for pair in dup_pairs[:300] for rid in pair}   # for the embedding sanity check
    pool = pool[pool["id"].isin(set(keep))].drop(columns=["_k_pair", "_k_instr"]).reset_index(drop=True)
    del lsh, texts

print(f"exact dups: {n_exact:,} | near dups: {len(dup_pairs):,} | pool now {len(pool):,}")
for a, b in dup_pairs[:3]:
    print("\nKEPT:   ", near_dup_texts[a][:150].replace("\n", " "), "\nDROPPED:", near_dup_texts[b][:150].replace("\n", " "))

---
## Stage 5 — Benchmark decontamination
Any training row sharing a **13-gram** (8-gram for short items) with a benchmark test item is removed —
from **every** arm, including A, so all arms are evaluated on equal terms. The held-out prompts are added
as a pseudo-benchmark, so nothing near them stays in the training pool.

Matching runs on prose only (LaTeX and digits removed): on the real data `frac 1 2 times …` and digit runs gave
false 13-gram hits, and so did FLAN-style prompt templates that hundreds of unrelated rows share. An n-gram that
occurs in more than `DECON_TEMPLATE_DF` pool rows (or in > 3 benchmark items) is treated as a template and ignored.

In [ ]:
from huggingface_hub import hf_hub_download

def tokens(s):
    return [t for t in WORD.findall(unicodedata.normalize("NFC", prose_only(s)).lower()) if not t.isdigit()]

def load_benchmarks():
    b = {}
    mm = pd.read_csv(hf_hub_download("openai/MMMLU", "test/mmlu_BN-BD.csv", repo_type="dataset"))
    b["mmlu_bn"] = (mm["Question"].astype(str) + " " + mm[["A", "B", "C", "D"]].astype(str).agg(" ".join, axis=1)).tolist()
    rqa = json.load(open(hf_hub_download("sartajekram/BanglaRQA", "Test.json", repo_type="dataset"), encoding="utf-8"))["data"]
    b["banglarqa_question"] = [q["question_text"] for p in rqa for q in p["qas"]]
    b["banglarqa_context"] = [p["context"] for p in rqa]
    b["mhumaneval_bn"] = pd.read_csv(CONFIG["MHUMANEVAL_BN_URL"])["prompt"].astype(str).tolist()
    for name, path, cols in CONFIG["EXTRA_BENCH_FILES"]:
        ext = pd.read_json(path, lines=path.endswith("l")) if ".json" in path else pd.read_csv(path)
        b[name] = ext[cols].astype(str).agg(" ".join, axis=1).tolist()
    b["heldout"] = heldout["instruction"].tolist()
    return b

def build_ngram_index(bench):
    """n -> {hash: benchmark}; n-grams found in more than 3 benchmark items are templates and skipped."""
    idx = {CONFIG["DECON_N"]: {}, CONFIG["DECON_SHORT_N"]: {}}
    item_df = Counter()
    for name, texts in bench.items():
        for t in texts:
            tk = tokens(t)
            n = CONFIG["DECON_N"] if len(tk) >= CONFIG["DECON_N"] else CONFIG["DECON_SHORT_N"]
            hs = {hash(tuple(tk[i:i + n])) for i in range(len(tk) - n + 1)}
            item_df.update(hs)
            for h in hs:
                idx[n].setdefault(h, name)
    return {n: {h: b for h, b in table.items() if item_df[h] <= 3} for n, table in idx.items()}

def matched_ngrams(text, idx):
    tk, hits = tokens(text), set()
    for n, table in idx.items():
        for i in range(len(tk) - n + 1):
            h = hash(tuple(tk[i:i + n]))
            if h in table:
                hits.add(h)
    return hits

with stage("decontaminate", len(pool)):
    BENCH = load_benchmarks()
    IDX = build_ngram_index(BENCH)
    lookup = {h: b for table in IDX.values() for h, b in table.items()}
    row_hits = [matched_ngrams(i + " " + o, IDX)
                for i, o in tqdm(zip(pool["instruction"], pool["output"]), total=len(pool), desc="decon")]
    pool_df = Counter(h for hs in row_hits for h in hs)          # pool rows sharing each matched n-gram
    pool["contaminated_by"] = [next((lookup[h] for h in hs if pool_df[h] <= CONFIG["DECON_TEMPLATE_DF"]), "")
                               for hs in row_hits]
    n_template = sum(c > CONFIG["DECON_TEMPLATE_DF"] for c in pool_df.values())
print({k: len(v) for k, v in BENCH.items()}, "| template n-grams ignored:", n_template)
print(pool["contaminated_by"].replace("", "clean").value_counts().to_string())

pool = pool[pool["contaminated_by"] == ""].drop(columns=["contaminated_by"]).reset_index(drop=True)
pool.to_parquet(OUT / "pool" / "pool_clean.parquet", index=False)
heldout.to_parquet(OUT / "pool" / "heldout.parquet", index=False)
print(f"CLEAN POOL: {len(pool):,} rows  (arm A = all of these)")

---
## Stages 6–8 need a GPU
Everything below is cached to `scores/*.parquet` chunk by chunk, so a Colab disconnect only loses the
current chunk — re-run the cell and it resumes.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("⚠ no GPU: Stages 6–8 will be very slow; use a GPU runtime (A100/L4) for anything beyond a smoke test")
# Native bf16 needs compute capability >= 8 (A100/L4). On T4 (sm_75) torch.cuda.is_bf16_supported() still says True
# because it counts software emulation, which made every model ~10x slower in the first Kaggle pilot -> use fp16 there.
BF16_OK = DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if BF16_OK else (torch.float16 if DEVICE == "cuda" else torch.float32)
print("dtype:", DTYPE)
PROMPT_TMPL = "### নির্দেশনা:\n{instruction}\n\n### উত্তর:\n"     # identical to the fine-tuning notebook

def cached_scores(name, frame, score_fn, chunk=2_000):
    path = OUT / "scores" / f"{name}.parquet"
    done = pd.read_parquet(path) if path.exists() else pd.DataFrame({"id": []})
    todo = frame[~frame["id"].isin(set(done["id"]))]
    for s in tqdm(range(0, len(todo), chunk), desc=name):
        res = score_fn(todo.iloc[s:s + chunk])
        done = pd.concat([done, res], ignore_index=True)
        done.to_parquet(path, index=False)
    return done[done["id"].isin(set(frame["id"]))]

def load_base_tokenizer():
    for name in (CONFIG["BASE_MODEL"], CONFIG["BASE_MODEL_FALLBACK"]):
        try:
            return name, AutoTokenizer.from_pretrained(name)
        except Exception as e:
            print(f"cannot load {name}: {type(e).__name__}")
    raise RuntimeError("no base tokenizer available")

BASE_NAME, base_tok = load_base_tokenizer()
pool["instr_tokens"] = [len(x) for x in base_tok(pool["instruction"].tolist(), add_special_tokens=False)["input_ids"]]
pool["resp_tokens"]  = [len(x) for x in base_tok(pool["output"].tolist(), add_special_tokens=False)["input_ids"]]
print(BASE_NAME, "| total tokens in pool:", f"{int(pool.instr_tokens.sum() + pool.resp_tokens.sum()):,}")

### Stage 6 — Perplexity and IFD (Cherry-LLM, Li et al. 2024)
With the *same base model we fine-tune*:
`IFD = PPL(response | instruction) / PPL(response)`. IFD ≥ 1 means the instruction does not help predict the
response (misaligned pair). High unconditional PPL within a domain flags garbled text. Both are reported and
offered as an extra baseline arm F; they are not used to drop rows from arm A.

In [ ]:
@torch.no_grad()
def response_nll(model, tok, prompts, responses, max_prompt=512, max_resp=512, bs=8):
    seqs, starts = [], []
    for p, r in zip(prompts, responses):
        p_ids = tok(p, add_special_tokens=False)["input_ids"][-max_prompt:] if p else []
        r_ids = tok(r, add_special_tokens=False)["input_ids"][:max_resp]
        seqs.append([tok.bos_token_id] + p_ids + r_ids)
        starts.append(1 + len(p_ids))
    order = np.argsort([len(s) for s in seqs])          # length-sorted batches = less padding
    out = np.full(len(seqs), np.nan)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]
        L = max(len(seqs[i]) for i in idx)
        ids = torch.full((len(idx), L), pad, dtype=torch.long)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for j, i in enumerate(idx):
            ids[j, :len(seqs[i])] = torch.tensor(seqs[i]); att[j, :len(seqs[i])] = 1
        logits = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits
        for j, i in enumerate(idx):
            st, en = starts[i], len(seqs[i])
            if en > st:
                out[i] = torch.nn.functional.cross_entropy(logits[j, st - 1:en - 1].float(), ids[j, st:en].to(DEVICE)).item()
    return out

if CONFIG["RUN_PPL_IFD"]:
    lm = AutoModelForCausalLM.from_pretrained(BASE_NAME, torch_dtype=DTYPE, device_map=DEVICE).eval()

    def score_ppl_ifd(part):
        prompts = [PROMPT_TMPL.format(instruction=i) for i in part["instruction"]]
        cond = response_nll(lm, base_tok, prompts, part["output"].tolist())
        uncond = response_nll(lm, base_tok, [""] * len(part), part["output"].tolist())
        return pd.DataFrame({"id": part["id"].values, "nll_cond": cond, "nll_uncond": uncond})

    with stage("ppl_ifd", len(pool)):
        ppl = cached_scores("ppl_ifd", pool, score_ppl_ifd)
    pool = pool.drop(columns=[c for c in ["nll_cond", "nll_uncond"] if c in pool]).merge(ppl, on="id", how="left")
    pool["ppl"] = np.exp(pool["nll_uncond"])
    pool["ifd"] = np.exp(pool["nll_cond"] - pool["nll_uncond"])
    pool["ppl_outlier"] = pool["ppl"] > pool.groupby("domain")["ppl"].transform(lambda s: s.quantile(0.99))
    print(f"IFD ≥ 1: {(pool.ifd >= 1).mean():.2%} | PPL outliers (top 1% per domain): {pool.ppl_outlier.sum():,}")
    display(pool.groupby("domain")[["ppl", "ifd"]].median().round(3))
    del lm; torch.cuda.empty_cache()

### Stage 7 — LLM-judge complexity *c* and quality *q* (DEITA-style)
The judge reads the Bengali text directly (no translation). Instead of parsing generated text we read the
probability the judge puts on each digit 1–6 for its **first token** and take the expectation, as DEITA's
scorers do. This gives a continuous score (few ties) and a `mass` value: if the digits get < 50% of the
probability, the judge did not follow the format and the score is marked unparseable.

In **pilot** mode both prompts are also run with a Bengali-language rubric; the Spearman correlation between
the English and Bengali rubric is the stability check from Task 4 (step 2).

In [ ]:
RUBRIC = {
 ("complexity", "en"): """You are rating Bengali (বাংলা) instructions for an instruction-tuning dataset.
Rate how COMPLEX / DEMANDING the instruction is, from 1 to 6:
1 = trivial (one fact, one-word answer, copying)
2 = simple (one step, common knowledge)
3 = moderate (a few steps or a short explanation)
4 = involved (multi-step reasoning, several constraints, or specialised knowledge)
5 = hard (deep reasoning, interacting constraints, long structured output)
6 = very hard (expert-level, careful multi-stage reasoning)
Judge only the difficulty of the task, not the writing quality. Do not translate it.

Instruction:
<<<
{instruction}
>>>

Reply with one digit from 1 to 6.""",
 ("quality", "en"): """You are rating a Bengali (বাংলা) instruction–response pair for an instruction-tuning dataset.
Rate the QUALITY of the RESPONSE from 1 to 6, considering:
- correctness and factual accuracy
- whether it follows every part of the instruction (format, length, all sub-questions)
- helpfulness and appropriate depth
- Bengali language quality: natural, fluent, grammatical; penalise machine-translation artefacts
  (literal English word order, untranslated English, broken conjuncts, wrong script)
1 = wrong, off-topic or unusable   2 = major errors   3 = partly correct or poorly written
4 = correct and acceptable   5 = correct, complete and fluent   6 = excellent, expert native-speaker quality

Instruction:
<<<
{instruction}
>>>

Response:
<<<
{response}
>>>

Reply with one digit from 1 to 6.""",
 ("complexity", "bn"): """নিচের বাংলা নির্দেশনাটি কতটা জটিল, ১ থেকে ৬ স্কেলে মূল্যায়ন করুন।
1 = খুব সহজ (একটি তথ্য বা এক শব্দের উত্তর)
2 = সহজ (এক ধাপের কাজ, সাধারণ জ্ঞান)
3 = মাঝারি (কয়েকটি ধাপ বা ছোট ব্যাখ্যা)
4 = জটিল (বহু-ধাপের যুক্তি, একাধিক শর্ত বা বিশেষ জ্ঞান)
5 = কঠিন (গভীর যুক্তি, পরস্পর-নির্ভর শর্ত, দীর্ঘ কাঠামোবদ্ধ উত্তর)
6 = অত্যন্ত কঠিন (বিশেষজ্ঞ-স্তরের বহু-ধাপের সমস্যা)
শুধু কাজটি কতটা কঠিন তা বিচার করুন, লেখার মান নয়।

নির্দেশনা:
<<<
{instruction}
>>>

শুধু 1 থেকে 6 এর মধ্যে একটি সংখ্যা লিখুন।""",
 ("quality", "bn"): """নিচের বাংলা নির্দেশনা ও উত্তরের জোড়াটি দেখে উত্তরের মান ১ থেকে ৬ স্কেলে মূল্যায়ন করুন। বিবেচনা করুন:
- তথ্য ও যুক্তির নির্ভুলতা
- নির্দেশনার প্রতিটি অংশ মানা হয়েছে কিনা (ফরম্যাট, দৈর্ঘ্য, সব উপ-প্রশ্ন)
- উত্তর কতটা সহায়ক ও যথেষ্ট গভীর
- বাংলা ভাষার মান: স্বাভাবিক, সাবলীল, ব্যাকরণগতভাবে শুদ্ধ; যান্ত্রিক অনুবাদের ছাপ (ইংরেজি বাক্যগঠন, অনূদিত না হওয়া ইংরেজি, ভাঙা যুক্তাক্ষর) থাকলে নম্বর কমান
1 = ভুল বা অপ্রাসঙ্গিক   2 = বড় ভুল আছে   3 = আংশিক সঠিক বা দুর্বল লেখা
4 = সঠিক ও গ্রহণযোগ্য   5 = সঠিক, সম্পূর্ণ ও সাবলীল   6 = চমৎকার, দক্ষ বাংলাভাষী বিশেষজ্ঞের মানের

নির্দেশনা:
<<<
{instruction}
>>>

উত্তর:
<<<
{response}
>>>

শুধু 1 থেকে 6 এর মধ্যে একটি সংখ্যা লিখুন।""",
}

def clip(s, n):
    return s if len(s) <= n else s[:n]

def clip_middle(s, n):        # keep the start and the final answer (math answers sit at the end)
    return s if len(s) <= n else s[: int(n * 0.7)] + "\n…\n" + s[-int(n * 0.3):]

def judge_messages(part, kind, lang):
    tmpl = RUBRIC[(kind, lang)]
    return [tmpl.format(instruction=clip(i, CONFIG["JUDGE_MAX_INSTR_CHARS"]),
                        response=clip_middle(o, CONFIG["JUDGE_MAX_RESP_CHARS"]))
            for i, o in zip(part["instruction"], part["output"])]

class DigitJudge:
    DIGITS = [str(d) for d in range(1, 7)]

    def __init__(self, model_name, backend):
        self.backend = backend
        if backend == "vllm":
            from vllm import LLM, SamplingParams
            # Kaggle 2×T4: no bf16 → fp16, and split the 7B judge across both GPUs
            self.llm = LLM(model=model_name, dtype="bfloat16" if BF16_OK else "float16",
                           tensor_parallel_size=max(1, torch.cuda.device_count()), max_model_len=8192,
                           gpu_memory_utilization=0.85, seed=CONFIG["SEED"])
            self.tok = self.llm.get_tokenizer()
            self.params = SamplingParams(max_tokens=1, temperature=0.0, logprobs=20)
        else:
            self.tok = AutoTokenizer.from_pretrained(model_name, padding_side="left")
            if self.tok.pad_token is None:
                self.tok.pad_token = self.tok.eos_token
            self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=DTYPE, device_map="auto").eval()
            self.digit_ids = {d: sorted({self.tok.encode(d, add_special_tokens=False)[-1],
                                         self.tok.encode(" " + d, add_special_tokens=False)[-1]}) for d in self.DIGITS}

    def _prompt(self, msg):
        return self.tok.apply_chat_template([{"role": "user", "content": msg}], tokenize=False, add_generation_prompt=True)

    def score(self, messages, bs=16, max_tokens=6_144):
        prompts = [self._prompt(m) for m in messages]
        dists = []
        if self.backend == "vllm":
            for o in self.llm.generate(prompts, self.params, use_tqdm=False):
                p = defaultdict(float)
                for tid, lp in (o.outputs[0].logprobs[0] if o.outputs[0].logprobs else {}).items():
                    tok_str = lp.decoded_token if getattr(lp, "decoded_token", None) is not None else self.tok.decode([tid])
                    if tok_str.strip() in self.DIGITS:
                        p[tok_str.strip()] += math.exp(lp.logprob)
                dists.append(p)
        else:
            # Longest-first batches capped at max_tokens padded tokens: little padding, and the
            # SDPA attention matrix (batch × heads × L²) fits next to a 7B judge on 2×T4.
            lens = [len(x) for x in self.tok(prompts, add_special_tokens=False)["input_ids"]]
            order = np.argsort(lens)[::-1]
            dists = [None] * len(prompts)
            b = 0
            while b < len(order):
                idx = order[b:b + max(1, min(bs, max_tokens // lens[order[b]]))]
                enc = self.tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
                with torch.no_grad():   # logits_to_keep=1: last position only, not batch × seq × 152K vocab
                    probs = self.model(**enc, logits_to_keep=1).logits[:, -1].float().softmax(-1).cpu()
                for i, row in zip(idx, probs):
                    dists[i] = {d: float(row[ids].sum()) for d, ids in self.digit_ids.items()}
                b += len(idx)
        mass = np.array([sum(p.values()) for p in dists])
        expected = np.array([sum(int(d) * v for d, v in p.items()) / m if m > 0 else np.nan for p, m in zip(dists, mass)])
        return expected, mass

judge = DigitJudge(CONFIG["JUDGE_MODEL"], CONFIG["JUDGE_BACKEND"])
LANGS = ["en", "bn"] if CONFIG["RUN_MODE"] == "pilot" else ["en"]

def score_judge(part):
    res = {"id": part["id"].values}
    for kind in ["complexity", "quality"]:
        for lang in LANGS:
            res[f"{kind}_{lang}"], res[f"{kind}_{lang}_mass"] = judge.score(judge_messages(part, kind, lang))
    return pd.DataFrame(res)

with stage("llm_judge", len(pool)):
    scores = cached_scores("judge_" + CONFIG["JUDGE_MODEL"].replace("/", "__"), pool, score_judge, chunk=1_000)
pool = pool.drop(columns=[c for c in scores.columns if c != "id" and c in pool]).merge(scores, on="id", how="left")

pool["parse_ok"] = (pool["complexity_en_mass"] > CONFIG["JUDGE_MIN_MASS"]) & (pool["quality_en_mass"] > CONFIG["JUDGE_MIN_MASS"])
pool["c"] = pool["complexity_en"]; pool["q"] = pool["quality_en"]
pool["s"] = pool["c"] * pool["q"]                      # DEITA: complexity × quality (not q²)

In [ ]:
# --- judge report (this is the Task 4 go / no-go evidence) ---
from scipy.stats import spearmanr

judge_report = {
    "parse_rate": round(float(pool["parse_ok"].mean()), 4),
    "c_mean": round(float(pool.c.mean()), 3), "q_mean": round(float(pool.q.mean()), 3),
    "spearman_c_vs_instr_len": round(spearmanr(pool.c, pool.instr_tokens, nan_policy="omit")[0], 3),
    "spearman_q_vs_resp_len":  round(spearmanr(pool.q, pool.resp_tokens, nan_policy="omit")[0], 3),
    "spearman_c_vs_q":         round(spearmanr(pool.c, pool.q, nan_policy="omit")[0], 3),
}
if "bn" in LANGS:
    judge_report["spearman_c_en_vs_bn"] = round(spearmanr(pool.complexity_en, pool.complexity_bn, nan_policy="omit")[0], 3)
    judge_report["spearman_q_en_vs_bn"] = round(spearmanr(pool.quality_en, pool.quality_bn, nan_policy="omit")[0], 3)
if "ifd" in pool:
    judge_report["spearman_q_vs_ifd"] = round(spearmanr(pool.q, pool.ifd, nan_policy="omit")[0], 3)
print(json.dumps(judge_report, indent=2))
display(pool.groupby("domain")[["c", "q", "s"]].mean().round(2))

# go / no-go thresholds from Task 4 (Part B)
if judge_report["parse_rate"] < 0.95:
    print("✗ judge does not follow the format reliably — change JUDGE_MODEL or the prompt")
    if CONFIG["RUN_MODE"] == "full":
        raise RuntimeError("judge parse rate too low for a full run; fix the judge in pilot mode first")
if judge_report.get("spearman_q_en_vs_bn", 1) < 0.5 or judge_report.get("spearman_c_en_vs_bn", 1) < 0.5:
    print("✗ English and Bengali rubrics disagree — the judge does not read Bengali stably")
if abs(judge_report["spearman_c_vs_instr_len"]) > 0.7:
    print("⚠ complexity is mostly instruction length — report token counts per arm (Task 3 risk)")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
pool.c.hist(bins=30, ax=ax[0]); ax[0].set_title("complexity c")
pool.q.hist(bins=30, ax=ax[1]); ax[1].set_title("quality q")
pool.s.hist(bins=30, ax=ax[2]); ax[2].set_title("s = c × q")
plt.tight_layout(); plt.savefig(OUT / "reports" / "judge_scores.png", dpi=120); plt.show()

del judge; torch.cuda.empty_cache()

### Stage 8 — Embeddings + sanity check
`bge-m3` (multilingual, covers Bengali). Before trusting the diversity step, check that near-duplicate pairs found
by MinHash are much closer than random pairs.

In [ ]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer(CONFIG["EMBED_MODEL"], device=DEVICE)
emb_model.max_seq_length = 512
if DEVICE == "cuda":
    emb_model.half()

def embed(texts):
    return emb_model.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True,
                            show_progress_bar=len(texts) > 5_000).astype(np.float32)

emb_path = OUT / "scores" / "embeddings.npy"
ids_path = OUT / "scores" / "embedding_ids.json"
with stage("embed", len(pool)):
    if emb_path.exists() and json.loads(ids_path.read_text()) == pool["id"].tolist():
        E = np.load(emb_path)
    else:
        E = embed((pool["instruction"] + "\n" + pool["output"].str[:1000]).tolist())
        np.save(emb_path, E); ids_path.write_text(json.dumps(pool["id"].tolist()))

# sanity check: MinHash near-dups vs random pairs
rng = np.random.default_rng(CONFIG["SEED"])
pairs = [(a, b) for a, b in dup_pairs if a in near_dup_texts and b in near_dup_texts][:200]
if pairs:
    A = embed([near_dup_texts[a] for a, _ in pairs]); B = embed([near_dup_texts[b] for _, b in pairs])
    dup_sim = (A * B).sum(1)
    i, j = rng.integers(0, len(E), 2_000), rng.integers(0, len(E), 2_000)
    rand_sim = (E[i] * E[j]).sum(1)
    print(f"near-dup pairs cos-sim: median {np.median(dup_sim):.3f} | random pairs: median {np.median(rand_sim):.3f}, "
          f"p95 {np.quantile(rand_sim, .95):.3f}, p99 {np.quantile(rand_sim, .99):.3f}")
    print(f"near-dups above random p99: {(dup_sim > np.quantile(rand_sim, .99)).mean():.1%}  (want > 90%)")
    print(f"pool pairs above τ={CONFIG['TAU_SIM']}: {(rand_sim >= CONFIG['TAU_SIM']).mean():.2%} of random pairs")

# nearest neighbours of 3 random rows — eyeball that "close" means "same task"
for q in rng.choice(len(E), 3, replace=False):
    nn = np.argsort(-(E @ E[q]))[1:3]
    print("\nQUERY:", pool.instruction[q][:120].replace("\n", " "))
    for k in nn:
        print(f"  {float(E[k] @ E[q]):.3f}", pool.instruction[k][:120].replace("\n", " "))

del emb_model; torch.cuda.empty_cache()

---
## Stage 9 — Build the experimental arms

| Arm | Size | Selection |
|---|---|---|
| A | all of the clean pool | no re-selection (TigerLLM anchor after hygiene + decontamination) |
| B | K | uniform random |
| C | K | sort by **s = c × q**, greedy add if cos-sim to nearest selected < τ (DEITA) |
| D | K | top-K by **q** only |
| E | K | top-K by **c × q**, no diversity step |
| F *(optional)* | K | top-K by IFD among IFD < 1 (Cherry-LLM baseline) |

Rows the judge could not score (`parse_ok == False`) are excluded from C–E (and reported).

In [ ]:
def deita_select(order, E, k, tau, chunk=2_048):
    """Exact greedy DEITA selection, vectorised per chunk. `order` = pool indices sorted by score (desc)."""
    Et = torch.from_numpy(E).to(DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32)
    sel_idx, sel_emb = [], Et[:0]
    for st in range(0, len(order), chunk):
        cand = np.asarray(order[st:st + chunk])
        C = Et[cand]
        prev_max = (C @ sel_emb.T).max(1).values.float().cpu().numpy() if len(sel_idx) else np.full(len(cand), -1.0)
        S = (C @ C.T).float().cpu().numpy()
        chosen = []
        for j in range(len(cand)):
            if prev_max[j] >= tau or (chosen and S[j, chosen].max() >= tau):
                continue
            chosen.append(j)
            if len(sel_idx) + len(chosen) >= k:
                break
        sel_idx += cand[chosen].tolist()
        sel_emb = torch.cat([sel_emb, C[chosen]])
        if len(sel_idx) >= k:
            break
    return sel_idx

K = min(CONFIG["SUBSET_SIZE"], len(pool))
rng = np.random.default_rng(CONFIG["SEED"])
jitter = rng.uniform(0, 1e-6, len(pool))                    # deterministic tie-break
scored = np.where(pool["parse_ok"].values)[0]
print(f"K = {K:,} | scorable rows: {len(scored):,} / {len(pool):,}")
if len(scored) < K:
    raise RuntimeError(f"only {len(scored):,} rows have a usable judge score but arms need {K:,}; see the judge report above")

ARMS = {"A": np.arange(len(pool)),
        "B": rng.choice(len(pool), K, replace=False)}

s_order = scored[np.argsort(-(pool["s"].values[scored] + jitter[scored]))]
def nn_similarity(E, chunk=4_096):
    """cos-sim of every row to its nearest other row (chunked: 100K × 100K never sits in memory)."""
    Et = torch.from_numpy(E).to(DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32)
    out = np.empty(len(E), np.float32)
    for st in range(0, len(E), chunk):
        S = Et[st:st + chunk] @ Et.T
        S[torch.arange(S.shape[0]), torch.arange(st, st + S.shape[0])] = -1
        out[st:st + chunk] = S.max(1).values.float().cpu().numpy()
    return out

NN_SIM = nn_similarity(E)
print("nearest-neighbour cos-sim quantiles:", {q: round(float(np.quantile(NN_SIM, q)), 3) for q in (.5, .9, .99)})
tau = (round(float(np.quantile(NN_SIM, CONFIG["TAU_NN_QUANTILE"])), 3)
       if CONFIG["TAU_SIM"] == "auto" else CONFIG["TAU_SIM"])
print("τ =", tau)
ARMS["C"] = deita_select(s_order, E, K, tau)
while len(ARMS["C"]) < K and tau < 0.99:                    # τ too strict for this pool → relax and log it
    tau = round(tau + 0.02, 2)
    print(f"C reached only {len(ARMS['C']):,} rows; relaxing τ to {tau}")
    ARMS["C"] = deita_select(s_order, E, K, tau)
TAU_USED = tau

ARMS["D"] = scored[np.argsort(-(pool["q"].values[scored] + jitter[scored]))][:K]
ARMS["E"] = s_order[:K]
if "ifd" in pool:
    ok = np.where(pool["parse_ok"].values & (pool["ifd"].values < 1))[0]
    ARMS["F"] = ok[np.argsort(-(pool["ifd"].values[ok] + jitter[ok]))][:K]

ARMS = {a: np.asarray(ix, dtype=np.int64) for a, ix in ARMS.items()}
print({a: len(ix) for a, ix in ARMS.items()}, "| τ used:", TAU_USED)
print("overlap C∩E:", f"{len(set(ARMS['C']) & set(ARMS['E'])) / K:.1%}",
      "| C∩D:", f"{len(set(ARMS['C']) & set(ARMS['D'])) / K:.1%}",
      "| C∩B:", f"{len(set(ARMS['C']) & set(ARMS['B'])) / K:.1%}")

In [ ]:
def distinct_n(texts, n=2, sample=3_000):
    texts = list(texts)[:sample]
    grams, total = set(), 0
    for t in texts:
        w = WORD.findall(t)
        g = list(zip(*[w[i:] for i in range(n)]))
        grams.update(g); total += len(g)
    return len(grams) / total if total else 0.0

probe = rng.choice(len(pool), min(3_000, len(pool)), replace=False)
Et_all = torch.from_numpy(E).to(DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32)

def arm_diag(name, ix):
    sub = pool.iloc[ix]
    S = Et_all[torch.from_numpy(ix).to(DEVICE)]
    coverage = (Et_all[torch.from_numpy(probe).to(DEVICE)] @ S.T).max(1).values.float().mean().item()
    pick = torch.from_numpy(rng.choice(len(ix), min(2_000, len(ix)), replace=False)).to(DEVICE)
    inner = (S[pick] @ S[pick].T).float()
    mean_pair_sim = ((inner.sum() - inner.diag().sum()) / (len(pick) * (len(pick) - 1))).item()
    shuffled = sub.sample(frac=1, random_state=CONFIG["SEED"])
    d = {"arm": name, "n": len(ix),
         "tokens_total": int(sub.instr_tokens.sum() + sub.resp_tokens.sum()),
         "resp_tokens_mean": round(sub.resp_tokens.mean(), 1), "instr_tokens_mean": round(sub.instr_tokens.mean(), 1),
         "c_mean": round(sub.c.mean(), 3), "q_mean": round(sub.q.mean(), 3),
         "distinct2_resp": round(distinct_n(shuffled.output), 4),
         "coverage": round(coverage, 4), "mean_pair_sim": round(mean_pair_sim, 4)}
    if "ifd" in sub:
        d["ifd_median"] = round(sub.ifd.median(), 3)
    d.update({f"dom_{k}": round(v, 3) for k, v in sub.domain.value_counts(normalize=True).items()})
    return d

diag = pd.DataFrame([arm_diag(a, ix) for a, ix in ARMS.items()]).fillna(0)
diag.to_csv(OUT / "reports" / "arm_diagnostics.csv", index=False)
display(diag.set_index("arm").T)
print("coverage = mean cos-sim from random pool rows to their nearest arm row (higher = arm spans the pool)")
print("mean_pair_sim = average similarity inside the arm (lower = more diverse)")

dom_cols = [c for c in diag.columns if c.startswith("dom_")]
diag.set_index("arm")[dom_cols].plot.bar(stacked=True, figsize=(10, 4), title="Domain mix per arm")
plt.legend(bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.savefig(OUT / "reports" / "arm_domains.png", dpi=120); plt.show()

In [ ]:
def to_record(r, arm):
    return {"id": r["id"], "arm": arm,
            "messages": [{"role": "user", "content": r["instruction"]}, {"role": "assistant", "content": r["output"]}],
            "instruction": r["instruction"], "output": r["output"], "domain": r["domain"],
            "c": None if pd.isna(r.get("c")) else round(float(r["c"]), 4),
            "q": None if pd.isna(r.get("q")) else round(float(r["q"]), 4)}

for arm, ix in ARMS.items():
    sub = pool.iloc[ix]
    with open(OUT / "arms" / f"arm_{arm}.jsonl", "w", encoding="utf-8") as f:
        for _, r in sub.iterrows():
            f.write(json.dumps(to_record(r, arm), ensure_ascii=False) + "\n")
with open(OUT / "arms" / "heldout.jsonl", "w", encoding="utf-8") as f:
    for _, r in heldout.iterrows():
        f.write(json.dumps(to_record(r, "heldout"), ensure_ascii=False) + "\n")
pool.to_parquet(OUT / "pool" / "pool_scored.parquet", index=False)
print("arms written ->", OUT / "arms")

---
## Stage 10 — Blind spot-check, cost extrapolation, manifest
`spotcheck_blind.csv` mixes 50 rows from C and 50 from B with the arm hidden; rate them by hand (or have a
teammate do it), then join with `spotcheck_key.csv`. This is Task 4 step 4 done blind, so it counts as evidence.

In [ ]:
n_sc = min(50, K)
sc = pd.concat([pool.iloc[rng.choice(ARMS["C"], n_sc, replace=False)].assign(arm="C"),
                pool.iloc[rng.choice(ARMS["B"], n_sc, replace=False)].assign(arm="B")])
sc = sc.sample(frac=1, random_state=CONFIG["SEED"]).reset_index(drop=True)
sc["item"] = np.arange(len(sc))
sc[["item", "instruction", "output"]].assign(rating_1to5="", notes="").to_csv(
    OUT / "reports" / "spotcheck_blind.csv", index=False, encoding="utf-8-sig")
sc[["item", "id", "arm", "c", "q"]].to_csv(OUT / "reports" / "spotcheck_key.csv", index=False)

# lowest-scored rows the judge rejected — do they deserve it?
for _, r in pool[pool.parse_ok].nsmallest(3, "s").iterrows():
    print(f"\n[s={r.s:.2f} c={r.c:.2f} q={r.q:.2f}] {r.instruction[:150]}\n→ {r.output[:200]}")

In [ ]:
scale = FULL_POOL_SIZE / max(1, len(pool))
est = {k: {"seconds": v["seconds"], "est_full_hours": round(v["seconds"] * scale / 3600, 2)}
       for k, v in TIMINGS.items() if k not in ("load", "domain_tag")}
manifest = {
    "config": CONFIG, "tau_used": TAU_USED, "base_model": BASE_NAME,
    "pool_rows_clean": len(pool), "heldout_rows": len(heldout),
    "near_dup_pairs": len(dup_pairs), "exact_dups": n_exact,
    "arms": {a: len(ix) for a, ix in ARMS.items()},
    "judge_report": judge_report, "timings": TIMINGS,
    "extrapolated_to_full_pool": est if CONFIG["RUN_MODE"] == "pilot" else None,
}
(OUT / "reports" / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(json.dumps(est, indent=2))